# 05 — SoilGrids uncertainty for the soil carbon stock

The headline stock (`02_carbon_pipeline.ipynb`) uses the SoilGrids `ocs` mean.
SoilGrids also publishes 5%, 50% and 95% quantiles of `ocs` 0–30 cm (Poggio et
al., 2021), but they are **not on Earth Engine**, so this notebook fetches them
from ISRIC's WCS for the district bounding box, samples them at every cell, and
propagates them to the district total.

**Why two bounds.** A district total needs the *correlation* of errors between
cells, which SoilGrids does not publish. Two limits bracket it:

| Bound | Assumption | Effect |
|---|---|---|
| Fully correlated | every cell is wrong in the same direction | Σ Q0.05 × area … Σ Q0.95 × area (widest) |
| Independent | cell errors cancel | normal approximation, sd per cell = (Q0.95 − Q0.05) / 3.29 (narrowest) |

SoilGrids errors come from covariates and model structure that vary smoothly
across a district, so the truth is much closer to the correlated bound. The
independent bound is shown to make clear how much a naive interval would
understate.

**Inputs.** `IIRS/Carbon_Stocks/soilgrids_quantiles/ocs_0-30cm_{Q0.05,Q0.5,Q0.95,mean}.tif`.
The first code cell downloads them if `maps.isric.org` is reachable; otherwise
download the four URLs it prints (any browser) into that folder and re-run.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import config as C

QDIR = C.DATA / "soilgrids_quantiles"
QDIR.mkdir(exist_ok=True)
LAYERS = ["Q0.05", "Q0.5", "Q0.95", "mean"]
BBOX = (75.30, 30.50, 76.40, 31.10)          # lon/lat, covers both district boundaries
WCS = ("https://maps.isric.org/mapserv?map=/map/ocs.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage"
       "&COVERAGEID=ocs_0-30cm_{q}&FORMAT=GEOTIFF_INT16"
       f"&SUBSET=X({BBOX[0]},{BBOX[2]})&SUBSET=Y({BBOX[1]},{BBOX[3]})"
       "&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326"
       "&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326")


In [ ]:
import urllib.request

def fetch(q):
    out = QDIR / f"ocs_0-30cm_{q}.tif"
    if out.exists() and out.stat().st_size > 1000:
        return "present"
    try:
        with urllib.request.urlopen(WCS.format(q=q), timeout=120) as r:
            data = r.read()
        if not data.startswith((b"II*", b"MM\x00*")):
            return "not a GeoTIFF: " + data[:120].decode(errors="replace")
        out.write_bytes(data)
        return f"downloaded {len(data):,} bytes"
    except Exception as e:
        return f"failed: {e}"

status = {q: fetch(q) for q in LAYERS}
for q, s in status.items():
    print(f"{q:6s} {s}")
missing = [q for q in LAYERS if not (QDIR / f"ocs_0-30cm_{q}.tif").exists()]
if missing:
    print("\nDownload these into", QDIR, "and re-run:")
    for q in missing:
        print(" ", WCS.format(q=q))
    raise SystemExit("SoilGrids quantile layers not available yet")


## Sample every cell

Nearest-pixel sampling at each cell centroid (SoilGrids is 250 m, the grid is
250 m). Mapped units of `ocs` are t/ha. That is checked, not assumed: the WCS
mean must reproduce the Earth Engine `ocs` already in the v2 table.


In [ ]:
import rasterio

cells = pd.read_csv(C.DATA / "v2" / "ludhiana_cells_v2.csv.gz",
                    usecols=["cell_id", "grid_row", "grid_col", "cell_area_ha", "in_district_frac",
                             f"in_{C.BOUNDARY_CHECK}", "soil_valid_frac", "ocs_030_t_ha"])
geo = C.lattice_geometry(cells["grid_row"], cells["grid_col"])
xy = list(zip(geo["lon"].astype(float), geo["lat"].astype(float)))

for q in LAYERS:
    with rasterio.open(QDIR / f"ocs_0-30cm_{q}.tif") as src:
        if src.crs is None or src.crs.to_epsg() != 4326:
            raise ValueError(f"{q}: expected EPSG:4326, got {src.crs}")
        v = np.array([s[0] for s in src.sample(xy)], dtype=float)
        if src.nodata is not None:
            v[v == src.nodata] = np.nan
    cells[f"wcs_{q}"] = v

ok = cells["soil_valid_frac"].ge(0.999) & cells["wcs_mean"].notna() & cells["ocs_030_t_ha"].notna()
r = cells.loc[ok, "wcs_mean"].corr(cells.loc[ok, "ocs_030_t_ha"])
ratio = float((cells.loc[ok, "wcs_mean"] / cells.loc[ok, "ocs_030_t_ha"]).median())
print(f"WCS mean vs Earth Engine ocs: r = {r:.3f}, median ratio = {ratio:.3f}  (n = {ok.sum():,})")
assert r > 0.9 and abs(ratio - 1) < 0.05, "WCS layer does not match the Earth Engine ocs: check units or georeferencing"
assert (cells.loc[ok, "wcs_Q0.05"] <= cells.loc[ok, "wcs_Q0.95"]).all()


## Propagate to the district total

In [ ]:
Z90 = 3.29                                   # width of a central 90% normal interval, in sd

def totals(area_in):
    a = area_in * cells["soil_valid_frac"]
    use = cells["wcs_mean"].notna() & cells["ocs_030_t_ha"].notna()
    a = a.where(use, 0)
    headline = float((cells["ocs_030_t_ha"] * a).sum())
    q05, q50, q95 = (float((cells[f"wcs_{q}"] * a).sum()) for q in ["Q0.05", "Q0.5", "Q0.95"])
    sd = (cells["wcs_Q0.95"] - cells["wcs_Q0.05"]) / Z90
    sd_ind = float(np.sqrt(((sd * a) ** 2).sum()))
    to_m = lambda t: round(t / 1e6, 3)
    return {
        "headline_mean_mtc": to_m(headline),
        "median_q50_mtc": to_m(q50),
        "correlated_90pct_mtc": [to_m(q05), to_m(q95)],
        "independent_90pct_mtc": [to_m(headline - 1.645 * sd_ind), to_m(headline + 1.645 * sd_ind)],
        "soil_area_ha": round(float(a.sum())),
    }

area = cells["cell_area_ha"]
out = {
    "source": "ISRIC SoilGrids 2.0 ocs 0-30cm Q0.05/Q0.5/Q0.95/mean, WCS maps.isric.org, sampled at cell centroids",
    "units_check": {"r_vs_earth_engine_ocs": round(r, 3), "median_ratio": round(ratio, 3)},
    "median_cell_relative_90pct_width": round(float(((cells["wcs_Q0.95"] - cells["wcs_Q0.05"]) / cells["wcs_mean"])[ok].median()), 3),
    C.BOUNDARY_DEFAULT: totals(area * cells["in_district_frac"]),
    C.BOUNDARY_CHECK: totals(area * cells[f"in_{C.BOUNDARY_CHECK}"]),
    "reading": ("SoilGrids does not publish error correlation. The correlated interval is the honest one to quote; "
                "the independent interval shows how far a naive sum would understate uncertainty."),
}
print(json.dumps(out, indent=2))
(C.RESULTS / "soilgrids_uncertainty.json").write_text(json.dumps(out, indent=2))
print("Wrote results/soilgrids_uncertainty.json; re-run 02_carbon_pipeline.ipynb to add it to the summary.")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
w = (cells["wcs_Q0.95"] - cells["wcs_Q0.05"])[ok]
ax[0].hist(w, bins=60, color="#3b6ea5")
ax[0].set(title="Per-cell 90% interval width, ocs 0–30 cm", xlabel="t/ha", ylabel="cells")
d = out[C.BOUNDARY_DEFAULT]
labels = ["Correlated", "Independent"]
for i, k in enumerate(["correlated_90pct_mtc", "independent_90pct_mtc"]):
    lo, hi = d[k]
    ax[1].plot([lo, hi], [i, i], lw=6, color=["#b5542d", "#3b6ea5"][i])
ax[1].axvline(d["headline_mean_mtc"], color="k", ls="--", label=f"headline {d['headline_mean_mtc']} MtC")
ax[1].set(yticks=[0, 1], yticklabels=labels, xlabel="MtC", title="District stock, 90% bounds")
ax[1].legend(); plt.tight_layout(); plt.savefig(C.RESULTS / "fig_soilgrids_uncertainty.png", dpi=130); plt.show()
